## Features that I want to have in this program:  
- have a database for my study record, maybe a excel file or csv file
- automatically save itself in git  
- show a estimate of day that I will acheive this 10_000 hours goal (based on my history)  
- show a calender that I can see my past activity  
- every day I can give one or more updates
- if previous day I didn't work, ask me to explain why I was away (also save this information in json file)  
- show a progress bar that I have acheive for example 50 percent of my goal  
- have env directory for required modules (run every where)

# TODO:
- take today's update
- automaticliy show today's data and time
- I want to do it OOP

In [1]:
import streamlit as st
import openpyxl
import subprocess
from datetime import datetime
from datetime import timedelta

class TenThousandHoursPython:
    def __init__(self):
        self.today = datetime.today()
        try:
            self.excel_file = openpyxl.load_workbook('data/daily_time_log.xlsx')
            print('Reports Excel file loaded')
        except FileNotFoundError:
            dirs = subprocess.check_output('ls -d */', shell=True, text=True)
            if 'data/' not in dirs:
                subprocess.run(['mkdir', 'data/'])
            self.excel_file = self.create_workbook()
            self.excel_file.save('data/daily_time_log.xlsx')
            print('New reports Excel file created')
        self.excel_file.save('data/daily_time_log.xlsx')
    
    def achievement(self):
        years = [i for i in range(self.today.year, self.today.year + 5)]
        # calculate for each year
        achievement_hours = 0   # accumolates after each year iteration
        total_days = 0          # accumolates after each year iteration
        for year in years:
            sheet = self.excel_file[f'{year}_report']
            non_empty_cells = []
            # find non-empty cells
            for i in range(2, 368):
                if sheet[f'B{i}'].value != None and sheet[f'B{i}'].value >= 0:
                    non_empty_cells.append(i)
            if non_empty_cells:      
                first_report_day = min(non_empty_cells)
                last_report_day = max(non_empty_cells)
                for day in range(first_report_day, last_report_day + 1):
                    if sheet[f'B{day}'].value == None:
                        sheet[f'B{day}'] = 0
                    total_days += 1
                    achievement_hours += sheet[f'B{day}'].value
            sheet['F1'] = achievement_hours                         # achievement hours
            sheet['F2'] = (achievement_hours /10_000) * 100         # achievement percent
            sheet['F3'] = (achievement_hours / total_days)          # mean hours per day
            sheet['F4'] = 10_000 - achievement_hours                # hours to end
            sheet['F5'] = sheet['F4'].value / sheet['F3'].value     # days to end
            sheet['F6'] = sheet['F5'].value / 7                     # weeks to end
            sheet['F7'] = sheet['F5'].value / 30                    # months to end
            sheet['F8'] = sheet['F5'].value / 365                   # years to end
            # if this year achieve move hours, then recalculate celebration day
            if year == years[0]:
                sheet['F9'] = (sheet[f'A{last_report_day}'].value + timedelta(sheet['F5'].value)).date()  # last celebration day based on accumolated hours
            elif self.excel_file[f'{year}_report']['F1'].value > self.excel_file[f'{year - 1}_report']['F1'].value:
                sheet['F9'] = (sheet[f'A{last_report_day}'].value + timedelta(sheet['F5'].value)).date()  # last celebration day based on accumolated hours
            else:
                sheet['F9'] = self.excel_file[f'{year - 1}_report']['F9'].value
        self.achievement_hours = achievement_hours
        self.achievement_percent = (achievement_hours /10_000) * 100
        self.mean_hours_per_day = (achievement_hours / total_days)
        self.hours_to_end = 10_000 - achievement_hours
        self.days_to_end = self.hours_to_end / self.mean_hours_per_day
        self.weeks_to_end = self.days_to_end / 7
        self.months_to_end = self.days_to_end / 30
        self.years_to_end = self.days_to_end / 365
        self.celebration_day = self.excel_file[f'{max(years)}_report']['F9'].value # last year, last result
        self.excel_file.save('data/daily_time_log.xlsx')

    def get_user_input(self):
        # get valid study or work hour
        while True:
            study_hours = input('How much did you study/work Python? ')
            try:
                float(study_hours)
            except ValueError:
                print('Enter a number')
                continue

            if 0<= float(study_hours) <= 18:
                if float(study_hours) % 0.5 == 0:
                    self.study_hours = float(study_hours)
                    break
                else:
                    print('Study/Work hour should be multiple of 0.5 (half hour)')
            else:
                print('Study/Work hour is not normal')
        # get valid user explanation report (max 60 character)
        while True:
            print('Explain in 60 character what did you do today?')
            print('Line shows 60 chracter length')
            print('*' * 60)
            explanation = input()
            if len(explanation) <= 60:
                self.explanation = explanation
                break
    
    def today_report(self):
        self.get_user_input()
        # find excel cell according to today and update its time
        sheet = self.excel_file[f'{self.today.year}_report']
        for cell in range(2,368):
            if str(self.today.date()) in str(sheet[f'A{cell}'].value):
                # if study hour is None , new value assign to it, else add to previous one 
                if sheet[f'B{cell}'].value == None:
                    sheet[f'B{cell}'] = float(self.study_hours)
                else:
                    sheet[f'B{cell}'] = sheet[f'B{cell}'].value + float(self.study_hours)
                sheet[f'C{cell}'] = self.explanation
        self.excel_file.save('data/daily_time_log.xlsx')

    def create_workbook(self):
        # create a new workbook
        excel_file = openpyxl.Workbook()
        # delete defualt sheet
        del excel_file['Sheet']
        # create five years from today
        years = [i for i in range(self.today.year, self.today.year + 5)]
        # default values of some cells in each sheet    
        cells = {
            'A1': 'date',
            'B1': 'study_work_hours',
            'C1': 'explanation',
            'E1': 'achievement_hours',
            'E2': 'achievement_percent',
            'E3': 'mean_hours_per_day',
            'E4': 'hours_to_end',
            'E5': 'days_to_end',
            'E6': 'weeks_to_end',
            'E7': 'months_to_end',
            'E8': 'years_to_end',
            'E9': 'celbration_day',
        }

        for year in years:
            excel_file.create_sheet(f'{year}_report')
            # change some cells to default values
            for cell, value in cells.items():
                excel_file[f'{year}_report'][cell] = value
            # find if year is leap year or not
            if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0):
                num_of_days = 366
            else:
                num_of_days = 365
            # create days of each year in A2:A366 or A2:A367 if year is leap
            for day in range(1, num_of_days + 1):
                # first access each sheet based on upper loop
                # then access each day cell based on this loop
                # asign datetime result to each cell
                # timedelta is -1 because start should be first day of year
                excel_file[f'{year}_report'][f'A{day + 1}'] = datetime(year=year, month=1, day=1) + timedelta(days=day - 1) 
        return excel_file






In [2]:
obj = TenThousandHoursPython()
obj.achievement()
# obj.today_report()
type(obj.achievement_hours)

Reports Excel file loaded


int

In [56]:
try:
    excel_file = openpyxl.load_workbook('data/daily_time_logs.xlsx')
except FileNotFoundError:
    print('Creating excel file')
# type(excel_file)
# excel_file.sheetnames
# excel_file['Sheet1'].title = '2026_record'
# excel_file.create_sheet('2027_record')


Creating excel file


In [51]:
excel_file.sheetnames
# excel_file.create_sheet('2027_record')
# excel_file.save('data/daily_time_log.xlsx')
# excel_file['Sheet1'].title = '2026_record'
# excel_file.sheetnames
# excel_file.create_sheet('2028_record', index=2)


['2026_record', '2027_record', '2028_record']

In [54]:
import os
# os.getcwd()
# os.listdir()
# os.mkdir("data")
# os.system('mv daily_time_log.xlsx data/')
# os.system('pip install openpyxl')
os.system('source env/bin/activate')



sh: 1: source: not found


32512

In [89]:
from datetime import datetime, timedelta
# date.today()
datetime.now().date()
datetime.now().time()
datetime.now() + timedelta(days=100)
diff = datetime(datetime.now().year, 12, 31) - datetime.now()
diff.days


324

In [110]:
# create template of excel file
# get todays year and make three sheets for three years including this year
import openpyxl
import subprocess
from datetime import datetime
from datetime import timedelta

obj = subprocess.check_output('ls -d */', shell=True, text=True)
if 'datas/' not in obj:
    subprocess.run(['mkdir', 'datas'])


In [120]:
datetime.today()

datetime.datetime(2026, 2, 9, 16, 23, 28, 143187)

In [145]:
file = openpyxl.Workbook()
del file['Sheet']

In [ ]:
today = datetime.today()

years = [today.year, today.year + 1, today.year + 2]

cells = {
    'A1': 'date',
    'B1': 'study_work_hours',
    'C1': 'explanation',
    'E1': 'achievement_hours',
    'E2': 'achievement_percent',
    'E3': 'mean_hours_per_day',
    'E4': 'hours_to_end',
    'E5': 'days_to_end',
    'E6': 'weeks_to_end',
    'E7': 'months_to_end',
    'E8': 'years_to_end',
    'E9': 'celbration_day',
}

for i in years:
    file.create_sheet(f'{i}_report')
    for cell, value in cells.items():
        file[f'{i}_report'][cell] = value
    # find if year is leap year or not
    if i % 4 == 0 and (i % 100 != 0 or i % 400 == 0):
        num_of_days = 366
    else:
        num_of_days = 365
    for day in num_of_days:
        file[f'{i}_report'][f'A{day + 1}'] = datetime(year=i, month=1, day=1) + timedelta(days=day - 1)  # to start from first day of first month
file.sheetnames

['2026_report', '2027_report', '2028_report']

In [173]:
def is_leap_year(year): 
    return year % 4 == 0 and (year % 100 != 0 or year % 400 == 0)
is_leap_year(2024)

True

In [ ]:
from datetime import timedelta
new_file = openpyxl.Workbook()

for i in range(1,366):
    new_file['Sheet'][f'A{i + 1}'] = (datetime(2027,1,1) + timedelta(days=i - 1))



new_file.save('test.xlsx')


In [155]:
cells = {
    'A1': 'date',
    'B1': 'study_work_hours',
    'C1': 'explanation',
    'E1': 'achievement_hours',
    'E2': 'achievement_percent',
    'E3': 'mean_hours_per_day',
    'E4': 'hours_to_end',
    'E5': 'days_to_end',
    'E6': 'weeks_to_end',
    'E7': 'months_to_end',
    'E8': 'years_to_end',
    'E9': 'celbration_day',
}

for cell, value in cells.items():
    file['2026_report'][cell] = value

In [281]:
file[-1]['A1':'C1']

KeyError: 'Worksheet -1 does not exist.'

In [168]:
print(datetime.today())

2026-02-09 17:17:09.171341


In [188]:
(datetime.today() + timedelta(1)).date()

datetime.date(2026, 2, 10)

In [233]:
file = openpyxl.load_workbook('test.xlsx')
# find non empty cells
for i in range(2, 367):
    if file['Sheet'][f'B{i}'].value != None and file['Sheet'][f'B{i}'].value > 0:
        print(file['Sheet'][f'B{i}'].value)


1
2
3
4


In [245]:
min([1,2,3,4])


1

In [248]:
bool([])

False

In [315]:
float('2.5') % 0.5 == 0.0


True

In [17]:
print('explain in 60 character what did you do today? line below is 60 character' )
print('*'* 60)

explain in 60 character what did you do today? line below is 60 character
************************************************************


In [4]:
num = '2.2'
float(num).isdigit()

AttributeError: 'float' object has no attribute 'isdigit'

In [26]:
userinput = '2.2'
try:
    float(userinput) 
    if float(userinput) % 0.5 == 0.0:
        print('valid')
except ValueError:
    print('not float')

In [22]:
float('2.2') % 0.5 == 0.0

False

In [34]:
userinput = '2.20'
0.0 <= float(userinput) <= 16

True

In [47]:
str(datetime.today().date()) in  str(datetime.today())

True

In [67]:
datetime.today().date()

datetime.date(2026, 2, 10)

In [ ]:
bool(datetime(2026, 12, 1))

SyntaxError: leading zeros in decimal integer literals are not permitted; use an 0o prefix for octal integers (939907410.py, line 1)

In [95]:
from  dateutil import relativedelta
from datetime import datetime
datetime(2026, 1, 1) + relativedelta(years=5)

TypeError: 'module' object is not callable

In [84]:
datetime.datetime(2026, 1, 1)

datetime.datetime(2026, 1, 1, 0, 0)

In [1]:
import subprocess
subprocess.run(['git', 'status'])

On branch master
Your branch is up to date with 'origin/master'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   data/daily_time_log.xlsx
	modified:   test.ipynb

no changes added to commit (use "git add" and/or "git commit -a")


CompletedProcess(args=['git', 'status'], returncode=0)

In [2]:
subprocess.run(['git', 'add', '.'])

CompletedProcess(args=['git', 'add', '.'], returncode=0)

In [3]:
subprocess.run(['git', 'commit', '-m "Autosave"'])

Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: empty ident name (for <happy@HAPPY.localdomain>) not allowed


CompletedProcess(args=['git', 'commit', '-m "Autosave"'], returncode=128)